# Landlab fundamentals: from a grid to a model

**Duration:** 2 hours 20 minutes, including a 10-minute break

This lesson is adapted from Ivy's `01-intro.ipynb`, `02-data.ipynb`,
`04-numerics.ipynb`, `05-components.ipynb`, and
`ESPIn_2026/01_a_simple_landlab_model.ipynb`.

By the end, you will be able to:

- create and inspect a raster grid;
- attach and visualize node fields;
- set boundary conditions;
- discover a component's field and parameter requirements;
- run `LinearDiffuser` in a time loop; and
- compare two model experiments.

Our reusable workflow is:

> question → grid → fields → boundaries → components → time loop → diagnostics

## 1. Meet the grid (20 minutes)

A Landlab grid represents the model domain. Nodes are points, links connect
adjacent nodes, and cells are areas surrounding interior nodes. We will work
with node data today. Landlab stores element data in flat arrays even when the
domain looks two-dimensional.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from landlab import RasterModelGrid, imshow_grid
from landlab.plot.graph import plot_graph

small_grid = RasterModelGrid((4, 5), xy_spacing=(10.0, 5.0))
plot_graph(small_grid, at="node,link,cell")

print(f"shape: {small_grid.shape}")
print(f"spacing: {small_grid.spacing}")
print(f"number of nodes: {small_grid.number_of_nodes}")
print(f"core nodes: {small_grid.core_nodes}")

### Predict, then inspect

Before running the next cell:

1. Which node is at `(x=20, y=5)`?
2. What shape do you expect `x_of_node` to have?
3. Why are perimeter nodes not core nodes?

In [ ]:
print("node IDs:", small_grid.nodes.reshape(small_grid.shape), sep="\n")
print("x coordinates:", small_grid.x_of_node)
print("y coordinates:", small_grid.y_of_node)
print("coordinate-array shape:", small_grid.x_of_node.shape)

## 2. Fields and boundaries (20 minutes)

A field is an array attached to a particular grid element. Components exchange
information through consistently named fields such as
`topographic__elevation`.

In [ ]:
grid = RasterModelGrid((25, 25), xy_spacing=10.0)
z = grid.add_zeros("topographic__elevation", at="node")

# Start with a gently sloping surface.
z[:] = 0.01 * grid.y_of_node

# Add a Gaussian mound using distances from the center node.
center_node = grid.grid_coords_to_node_id(12, 12)
distance = np.hypot(
    grid.x_of_node - grid.x_of_node[center_node],
    grid.y_of_node - grid.y_of_node[center_node],
)
z += 50.0 * np.exp(-(distance**2) / (2.0 * 25.0**2))

print("node fields:", list(grid.at_node))
print("field shape:", z.shape)
imshow_grid(grid, "topographic__elevation", cmap="terrain", colorbar_label="Elevation (m)")

Boundary nodes do not update like core nodes. Here, sediment cannot cross the
left and right edges; the top and bottom remain fixed-value boundaries. Before
running the cell, predict which directions allow material to leave the domain.

In [ ]:
grid.set_closed_boundaries_at_grid_edges(
    right_is_closed=True,
    top_is_closed=False,
    left_is_closed=True,
    bottom_is_closed=False,
)

print("core nodes:", grid.number_of_core_nodes)
print("closed boundary nodes:", np.count_nonzero(grid.status_at_node == grid.BC_NODE_IS_CLOSED))
print("fixed-value boundary nodes:", np.count_nonzero(grid.status_at_node == grid.BC_NODE_IS_FIXED_VALUE))

### Pair exercise

Choose one change, predict its effect, make it, and rerun the plot:

- double the mound height;
- move the mound center;
- reverse the background slope; or
- close a different pair of boundaries.

Restore and rerun the supplied cells before continuing.

## 3. Components are contracts (15 minutes)

A Landlab component is a Python class representing a process or calculation.
It expects fields with particular names, locations, and units. Its metadata and
docstring are often the fastest way to understand how to use it.

In [ ]:
from landlab.components import LinearDiffuser

print("inputs:", LinearDiffuser.input_var_names)
print("outputs:", LinearDiffuser.output_var_names)
LinearDiffuser.var_help("topographic__elevation")

### Documentation challenge

Run `help(LinearDiffuser)` or place the cursor inside
`LinearDiffuser(...)` and press **Shift+Tab**. Find:

1. the required grid argument;
2. the parameter controlling diffusivity; and
3. the method that advances the component through time.

In [ ]:
help(LinearDiffuser)

## Break (10 minutes)

## 4. Run the first model (30 minutes)

The model state already lives in `grid.at_node`. Instantiating a component does
not advance time; the loop below does that explicitly. We retain the initial
surface so that the model result can be evaluated, not merely displayed.

In [ ]:
z_initial = z.copy()
diffuser = LinearDiffuser(grid, linear_diffusivity=0.02)

time_step = 10.0  # years
total_time = 10_000.0  # years
elapsed_time = 0.0

while elapsed_time < total_time:
    diffuser.run_one_step(time_step)
    elapsed_time += time_step

print(f"completed {elapsed_time:,.0f} years")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plt.sca(axes[0])
imshow_grid(grid, z_initial, cmap="terrain", colorbar_label="Elevation (m)")
axes[0].set_title("Initial")
plt.sca(axes[1])
imshow_grid(grid, z, cmap="terrain", colorbar_label="Elevation (m)")
axes[1].set_title("Final")
plt.sca(axes[2])
imshow_grid(grid, z - z_initial, cmap="RdBu", colorbar_label="Elevation change (m)")
axes[2].set_title("Change")
plt.tight_layout()

print(f"initial relief: {np.ptp(z_initial):.2f} m")
print(f"final relief:   {np.ptp(z):.2f} m")

### Interpret before continuing

- Where was material removed and where was it deposited?
- Did relief increase or decrease?
- How did the boundary conditions influence the edges?
- Which line would you change to represent a more diffusive landscape?

## 5. A controlled experiment (20 minutes)

Scientific comparison is easier when setup and execution are repeatable. The
function below packages the same model so that only diffusivity changes.

In [ ]:
def run_mound_diffusion(diffusivity, total_time=10_000.0, time_step=10.0):
    experiment_grid = RasterModelGrid((25, 25), xy_spacing=10.0)
    elevation = experiment_grid.add_zeros("topographic__elevation", at="node")
    elevation[:] = 0.01 * experiment_grid.y_of_node

    center = experiment_grid.grid_coords_to_node_id(12, 12)
    distance = np.hypot(
        experiment_grid.x_of_node - experiment_grid.x_of_node[center],
        experiment_grid.y_of_node - experiment_grid.y_of_node[center],
    )
    elevation += 50.0 * np.exp(-(distance**2) / (2.0 * 25.0**2))
    experiment_grid.set_closed_boundaries_at_grid_edges(True, False, True, False)

    initial = elevation.copy()
    component = LinearDiffuser(experiment_grid, linear_diffusivity=diffusivity)
    for _ in range(int(total_time / time_step)):
        component.run_one_step(time_step)

    return experiment_grid, initial, elevation


slow_grid, slow_initial, slow_final = run_mound_diffusion(0.01)
fast_grid, fast_initial, fast_final = run_mound_diffusion(0.05)

print(f"D=0.01 final relief: {np.ptp(slow_final):.2f} m")
print(f"D=0.05 final relief: {np.ptp(fast_final):.2f} m")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plt.sca(axes[0])
imshow_grid(slow_grid, slow_final, cmap="terrain", colorbar_label="Elevation (m)")
axes[0].set_title("D = 0.01")
plt.sca(axes[1])
imshow_grid(fast_grid, fast_final, cmap="terrain", colorbar_label="Elevation (m)")
axes[1].set_title("D = 0.05")
plt.tight_layout()

### Pair experiment

1. Predict what changing one variable will do.
2. Change **one** of diffusivity, model duration, grid spacing, initial
   topography, or boundaries.
3. Compare against the supplied baseline.
4. Write one claim supported by a plot or relief value.

**Our prediction:**  
**What we changed:**  
**Evidence:**  
**Interpretation:**

## 6. From synthetic terrain to supplied data (15 minutes)

Real data introduce coordinate systems, missing values, resolution, and
runtime. During this introduction we use a local ESRI ASCII file and inspect
it. Downloading and reprojection are project-specific extensions.

In [ ]:
from pathlib import Path
from landlab.io import read_esri_ascii

dem_path = Path("../landlab/DEMData/SRTMGL1_39.93_-105.33_40.0_-105.26.asc")
dem_grid, dem_z = read_esri_ascii(dem_path, name="topographic__elevation")

print(f"shape: {dem_grid.shape}")
print(f"spacing in source file: {dem_grid.spacing}")
print(f"elevation range: {dem_z.min():.1f} to {dem_z.max():.1f} m")
imshow_grid(dem_grid, "topographic__elevation", cmap="terrain", colorbar_label="Elevation (m)")

The source grid coordinates are geographic degrees, so a distance-based model
must use a projected grid with internally consistent horizontal and vertical
units. For today's short project, use a supplied projected setup or synthetic
grid unless reprojection is itself your learning goal.

## Exit check

Without looking back, explain to a partner:

1. where the model state is stored;
2. what a component needs before it can run;
3. what advances model time; and
4. the first diagnostic you would use to decide whether a run is plausible.

Next, open one advanced track. Both tracks culminate in coupling components.

---

# Core lesson ends here

You now have everything required for either advanced track. The following
sections are optional extensions for participants who finish early or want to
see more of the numerical machinery. They are **not prerequisites** for the
advanced session and do not introduce component coupling.

## Optional A — Under the hood: build diffusion yourself (35–50 minutes)

`LinearDiffuser` hides a sequence of grid calculations. Here we reconstruct a
simple explicit diffusion model using links, gradients, fluxes, and flux
divergence.

The governing relationships are

$$q_s = -D \nabla z$$

and

$$\frac{\partial z}{\partial t} = -\nabla \cdot q_s,$$

where elevation $z$ is stored at nodes, its gradient and sediment flux $q_s$
are stored at links, and flux divergence is calculated back at nodes.

In [ ]:
def make_mound_grid(spacing=10.0):
    # Create the same reproducible initial condition for model comparisons.
    model_grid = RasterModelGrid((25, 25), xy_spacing=spacing)
    elevation = model_grid.add_zeros("topographic__elevation", at="node")
    elevation[:] = 0.01 * model_grid.y_of_node

    center = model_grid.grid_coords_to_node_id(12, 12)
    distance = np.hypot(
        model_grid.x_of_node - model_grid.x_of_node[center],
        model_grid.y_of_node - model_grid.y_of_node[center],
    )
    elevation += 50.0 * np.exp(-(distance**2) / (2.0 * 25.0**2))
    model_grid.set_closed_boundaries_at_grid_edges(True, False, True, False)
    return model_grid, elevation


manual_grid, manual_z = make_mound_grid()

print("first ten links as [tail node, head node]:")
print(manual_grid.nodes_at_link[:10])
print("first ten link lengths:")
print(manual_grid.length_of_link[:10])

### Follow one calculation across the grid

Before running the next cell, predict:

1. the length of the node-gradient array;
2. where the steepest positive and negative gradients occur; and
3. why the sediment flux has the opposite sign to the elevation gradient.

In [ ]:
diffusivity = 0.02
gradient_at_link = manual_grid.calc_grad_at_link(manual_z)
sediment_flux = manual_grid.zeros(at="link")
sediment_flux[manual_grid.active_links] = (
    -diffusivity * gradient_at_link[manual_grid.active_links]
)
rate_of_elevation_change = -manual_grid.calc_flux_div_at_node(sediment_flux)

print(f"node values: {manual_grid.number_of_nodes}")
print(f"link gradients: {gradient_at_link.size}")
print(f"maximum |gradient|: {np.abs(gradient_at_link).max():.3f}")
print(f"maximum |dz/dt|: {np.abs(rate_of_elevation_change).max():.5f}")

### Choose a stable timestep

For this explicit calculation, use a conservative Courant-style criterion:

$$\Delta t = 0.2rac{\Delta x_{min}^2}{D}.$$

Try multiplying the stable timestep by 10 after completing the exercise. What
symptom tells you that a numerical result has become unstable?

In [ ]:
stable_time_step = 0.2 * manual_grid.length_of_link.min() ** 2 / diffusivity
total_time = 10_000.0
number_of_steps = int(np.ceil(total_time / stable_time_step))
actual_time_step = total_time / number_of_steps
manual_initial = manual_z.copy()

print(f"stable timestep: {stable_time_step:.1f} years")
print(f"using {number_of_steps} steps of {actual_time_step:.1f} years")

for _ in range(number_of_steps):
    gradient_at_link = manual_grid.calc_grad_at_link(manual_z)
    sediment_flux[manual_grid.active_links] = (
        -diffusivity * gradient_at_link[manual_grid.active_links]
    )
    rate_of_elevation_change = -manual_grid.calc_flux_div_at_node(sediment_flux)
    manual_z[manual_grid.core_nodes] += (
        rate_of_elevation_change[manual_grid.core_nodes] * actual_time_step
    )

In [ ]:
component_grid, component_z = make_mound_grid()
component = LinearDiffuser(component_grid, linear_diffusivity=diffusivity)
for _ in range(number_of_steps):
    component.run_one_step(actual_time_step)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plt.sca(axes[0])
imshow_grid(manual_grid, manual_z, cmap="terrain", colorbar_label="Elevation (m)")
axes[0].set_title("Manual grid operations")
plt.sca(axes[1])
imshow_grid(component_grid, component_z, cmap="terrain", colorbar_label="Elevation (m)")
axes[1].set_title("LinearDiffuser")
plt.sca(axes[2])
imshow_grid(
    manual_grid,
    manual_z - component_z,
    cmap="RdBu",
    colorbar_label="Elevation difference (m)",
)
axes[2].set_title("Manual minus component")
plt.tight_layout()

print(f"mean absolute difference: {np.mean(np.abs(manual_z - component_z)):.6f} m")

### Explain the data movement

Draw this sequence and annotate each quantity with its grid element:

> node elevation → link gradient → link flux → node divergence → node elevation

Then answer:

- Why do we update only `core_nodes`?
- Why do we calculate flux only on `active_links`?
- Which parts of this implementation does `LinearDiffuser` save you from
  maintaining?

## Optional B — Python challenge: parameter sweeps (20–30 minutes)

A final map can hide how quickly a system changed. Run several diffusivities,
record relief through time, and compare all trajectories. This exercise uses
ordinary Python functions, dictionaries, and plotting around one familiar
Landlab component.

In [ ]:
def relief_through_time(diffusivity, total_time=10_000.0, time_step=10.0):
    experiment_grid, elevation = make_mound_grid()
    diffuser = LinearDiffuser(experiment_grid, linear_diffusivity=diffusivity)

    times = [0.0]
    relief = [np.ptp(elevation)]
    elapsed = 0.0
    record_every = 50

    for step in range(1, int(total_time / time_step) + 1):
        diffuser.run_one_step(time_step)
        elapsed += time_step
        if step % record_every == 0:
            times.append(elapsed)
            relief.append(np.ptp(elevation))

    return np.asarray(times), np.asarray(relief)


diffusivities = [0.005, 0.01, 0.02, 0.05]
experiments = {
    diffusivity: relief_through_time(diffusivity)
    for diffusivity in diffusivities
}

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for diffusivity, (times, relief) in experiments.items():
    ax.plot(times, relief, label=f"D = {diffusivity:g}")

ax.set(
    xlabel="Time (years)",
    ylabel="Topographic relief (m)",
    title="Diffusion reduces relief through time",
)
ax.legend()
ax.grid(alpha=0.25)

### Extension challenge

Choose one:

1. Find the first time at which each run falls below 25 m of relief.
2. Repeat the sweep with a different grid spacing. Use the stable-timestep
   relationship from Optional A to explain the computational effect.
3. Store final elevation as well as relief and create a comparison figure.

Write one claim that is supported by the trajectories rather than by the model
equations alone.

These exercises complete the optional material. Continue to one advanced track
for multi-component coupling and application-specific diagnostics.